<hr>

#### <strong>第四次作品：去除影像模糊的深度學習實驗</strong>

學號：411278018  
姓名：（Moses）  
日期：2026-05-17


<hr>

<strong><font color="darkgoldenrod">作品目標</font></strong>

本作品以 SRCNN（Super-Resolution Convolutional Neural Network）為出發點，檢驗其用於影像去模糊（Image Deblurring）的可行性。實驗不僅重現教授示範中的三層卷積基線模型，也進一步比較加入 Batch Normalization 與增加網路深度後的效果，藉由相同資料切分、相同損失函數與相同優化器設定，使三種模型的差異能夠集中反映在架構設計本身。評估指標採用 PSNR（Peak Signal-to-Noise Ratio），並以 Set5 與 Set14 作為公開測試影像集合，以便從訓練表現、驗證表現與測試影像復原品質三個面向觀察模型能力。

<table>
<thead><tr><th>項目</th><th>教授示範版本</th><th>本作品版本</th></tr></thead>
<tbody>
<tr><td>訓練影像數量</td><td>General100，共 100 張</td><td><span style="color: darkgreen;">T91 加 General100，共 191 張</span></td></tr>
<tr><td>影像處理策略</td><td>全圖 resize 至 224×224</td><td><span style="color: darkgreen;">64×64 隨機 patch crop，每圖 10 個 sampling units</span></td></tr>
<tr><td>模糊影像產生方式</td><td>離線預先計算並存入目錄</td><td><span style="color: darkgreen;">DataLoader 讀取時動態產生，不預存模糊影像</span></td></tr>
<tr><td>Model B</td><td>DeblurCNN_RES residual connection</td><td><span style="color: darkgreen;">DeblurCNN_BN，使用 BatchNorm 標準化</span></td></tr>
<tr><td>Model C</td><td>DeblurSuperResCNN，feature concat 加 Sigmoid</td><td><span style="color: darkgreen;">DeblurDeep5，五層均勻卷積且輸出層不加 Sigmoid</span></td></tr>
<tr><td>學習率排程</td><td>ReduceLROnPlateau 監控驗證損失</td><td><span style="color: darkgreen;">CosineAnnealingLR 進行週期性退火</span></td></tr>
<tr><td>模型評估</td><td>單模型訓練曲線</td><td><span style="color: darkgreen;">三模型 PSNR 比較與 Set5、Set14 系統測試</span></td></tr>
</tbody>
</table>


<hr>

<strong><font color="darkgoldenrod">實驗計畫</font></strong>

本實驗使用 T91 與 General100 兩組超解析度領域常見資料集作為訓練來源，合計 191 張高解析度影像；每張影像對應 10 個 64×64 patch sampling units，因此有效樣本數為 1,910，並以 9:1 比例切分為 1,719 個訓練 sampling units 與 191 個驗證 sampling units。模糊策略採用高斯模糊，標準差固定為 σ = 3，kernel size 交由 OpenCV 依據 σ 自動計算，且模糊影像在資料讀取時即時產生，不另行儲存於磁碟，以避免不必要的資料重複與路徑管理負擔。

模型設計分為三個層次。Model A 為教授示範中相同的三層 SRCNN 架構，作為本實驗的基線；Model B 在三層卷積後加入 BatchNorm，使每層輸出分布較穩定，理論上可加速收斂並減緩梯度傳遞不穩定，且此處採用 BatchNorm 而非示範中的 residual connection；Model C 將網路深度增加至五層，channel 配置為 3→64→128→64→32→3，並使用均勻的 3×3 kernel，以提高局部紋理復原能力，同時取消示範中 DeblurSuperResCNN 的 feature concatenation 與輸出層 Sigmoid。三個模型皆以 MSELoss 為損失函數，使用 Adam 優化器，學習率為 0.001，批次大小為 32，訓練 50 個 epoch，並透過 CosineAnnealingLR 在 50 個 epoch 內進行學習率退火。訓練過程逐 epoch 記錄 Train 與 Val PSNR，訓練完成後再以 Set5 與 Set14 共 19 張影像進行測試，最後以一張自選影像進行定性比較。


<hr>

<strong><font color="darkgoldenrod">套件載入</font></strong>

本節匯入影像處理、深度學習、資料分割、視覺化與進度顯示所需套件，並先檢查 PyTorch 版本與可用運算裝置。


In [ ]:
import os
import math
import random
from pathlib import Path

import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import pandas as pd

from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(
    "Using device:",
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu",
)


In [ ]:
BASE_DIR = Path().resolve()
DATA_DIR = BASE_DIR / "DeblurCNN_2026" / "inputs"
OUTPUT_DIR = BASE_DIR / "DeblurCNN_2026" / "outputs"
SET5_DIR = DATA_DIR / "Set5"
SET14_DIR = DATA_DIR / "Set14"
TRAIN_DIRS = [DATA_DIR / "T91", DATA_DIR / "General100"]
CUSTOM_IMG_PATH = BASE_DIR / "your_photo.jpg"

PATCH_SIZE = 64
N_PATCHES = 10
SIGMA = 3.0
BATCH_SIZE = 32
LR = 0.001
NUM_EPOCHS = 50
DEMO_EPOCHS = 2
RANDOM_STATE = 42

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR  : {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Device    : {device}")


<hr>

<strong><font color="darkgoldenrod">共用函數定義</font></strong>

本節定義整個影像去模糊實驗共同使用的評估函數、模糊函數與 patch 資料集類別。PSNR 用以衡量復原影像與參考影像之間的峰值訊噪比，數值愈高代表失真愈小；在一般影像復原任務中，30 dB 以上通常可視為具備可接受的重建品質。相較於將全圖 resize 至固定尺寸，patch-based 訓練能保留局部紋理與邊緣細節，同時把有限的影像數量轉化為更多有效訓練樣本，使模型在資料規模較小時仍能學習多樣的局部模糊與復原關係。

$$\text{PSNR}(\hat{I}, I) = 20 \cdot \log_{10}\!\left(\frac{\text{MAX}_I}{\sqrt{\frac{1}{N}\sum_{n=1}^{N}(\hat{I}_n - I_n)^2}}\right)$$

其中 $\text{MAX}_I = 1.0$，表示影像已正規化至 $[0,1]$，而 $N$ 為像素總數。


In [ ]:
def psnr(pred, target, max_val=1.0) -> float:
    if isinstance(pred, torch.Tensor):
        pred_np = pred.detach().cpu().numpy()
    else:
        pred_np = np.asarray(pred)

    if isinstance(target, torch.Tensor):
        target_np = target.detach().cpu().numpy()
    else:
        target_np = np.asarray(target)

    if pred_np.shape != target_np.shape:
        raise ValueError("pred and target must have the same shape")

    diff = pred_np.astype(np.float64) - target_np.astype(np.float64)
    rmse = math.sqrt(np.mean(diff**2))
    if rmse == 0:
        return 100.0
    return float(20 * math.log10(max_val / rmse))


def apply_blur(img_rgb, sigma=3) -> np.ndarray:
    if not isinstance(img_rgb, np.ndarray):
        raise TypeError("img_rgb must be a numpy array")
    if img_rgb.ndim != 3 or img_rgb.shape[2] != 3:
        raise ValueError("img_rgb must have shape (H, W, 3)")
    if img_rgb.dtype != np.uint8:
        raise TypeError("img_rgb must have dtype uint8")
    return cv2.GaussianBlur(img_rgb, (0, 0), sigmaX=float(sigma))


class DeblurPatchDataset(Dataset):
    def __init__(
        self,
        image_paths: list[Path],
        patch_size: int,
        n_patches: int,
        sigma: float,
        transform,
    ):
        if patch_size <= 0:
            raise ValueError("patch_size must be positive")
        if n_patches <= 0:
            raise ValueError("n_patches must be positive")
        if not image_paths:
            raise ValueError("image_paths must not be empty")

        self.image_paths = [Path(path) for path in image_paths]
        self.patch_size = int(patch_size)
        self.n_patches = int(n_patches)
        self.sigma = float(sigma)
        self.transform = transform if transform is not None else transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths) * self.n_patches

    def __getitem__(self, idx):
        image_idx = idx // self.n_patches
        image_path = self.image_paths[image_idx]
        img_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        height, width = img_rgb.shape[:2]
        if height < self.patch_size or width < self.patch_size:
            scale = max(self.patch_size / height, self.patch_size / width)
            new_width = math.ceil(width * scale)
            new_height = math.ceil(height * scale)
            img_rgb = cv2.resize(
                img_rgb,
                (new_width, new_height),
                interpolation=cv2.INTER_CUBIC,
            )
            height, width = img_rgb.shape[:2]

        top = random.randint(0, height - self.patch_size)
        left = random.randint(0, width - self.patch_size)
        sharp_patch = img_rgb[
            top : top + self.patch_size,
            left : left + self.patch_size,
            :,
        ]
        blur_patch = apply_blur(sharp_patch, self.sigma)

        sharp_patch = sharp_patch.astype(np.float32) / 255.0
        blur_patch = blur_patch.astype(np.float32) / 255.0

        blur_tensor = self.transform(blur_patch).float()
        sharp_tensor = self.transform(sharp_patch).float()
        return blur_tensor, sharp_tensor


print("共用函數已定義：psnr, apply_blur, DeblurPatchDataset")


In [ ]:
class DeblurCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, stride=1, padding=2)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=1, stride=1, padding=2)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, stride=1, padding=2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x


class DeblurCNN_BN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=1, padding=0)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, padding=2)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.conv3(x)
        return x


class DeblurDeep5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(32)
        self.conv5 = nn.Conv2d(32, 3, kernel_size=3, padding=1)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.conv5(x)
        return x


def count_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def train_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int,
    device: str,
    output_dir: Path,
    scheduler_cls,
    scheduler_kwargs: dict,
) -> dict:
    if epochs <= 0:
        raise ValueError("epochs must be positive")
    if len(train_loader.dataset) == 0 or len(val_loader.dataset) == 0:
        raise ValueError("train_loader and val_loader must contain data")

    output_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = scheduler_cls(optimizer, **scheduler_kwargs)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_psnr": [],
        "val_psnr": [],
    }
    best_val_psnr = -math.inf

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_total = 0.0
        train_psnr_values = []

        progress = tqdm(
            train_loader,
            desc=f"{model_name} {epoch:03d}/{epochs:03d}",
            leave=False,
        )
        for blur_batch, sharp_batch in progress:
            blur_batch = blur_batch.to(device)
            sharp_batch = sharp_batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(blur_batch)
            loss = criterion(outputs, sharp_batch)
            loss.backward()
            optimizer.step()

            batch_size = blur_batch.size(0)
            train_loss_total += loss.item() * batch_size
            train_psnr_values.append(psnr(outputs.clamp(0, 1), sharp_batch))

        model.eval()
        val_loss_total = 0.0
        val_psnr_values = []
        with torch.no_grad():
            for blur_batch, sharp_batch in val_loader:
                blur_batch = blur_batch.to(device)
                sharp_batch = sharp_batch.to(device)
                outputs = model(blur_batch)
                loss = criterion(outputs, sharp_batch)

                batch_size = blur_batch.size(0)
                val_loss_total += loss.item() * batch_size
                val_psnr_values.append(psnr(outputs.clamp(0, 1), sharp_batch))

        scheduler.step()

        train_loss = train_loss_total / len(train_loader.dataset)
        val_loss = val_loss_total / len(val_loader.dataset)
        train_psnr_epoch = float(np.mean(train_psnr_values))
        val_psnr_epoch = float(np.mean(val_psnr_values))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_psnr"].append(train_psnr_epoch)
        history["val_psnr"].append(val_psnr_epoch)

        print(
            f"Epoch {epoch:03d}/{epochs:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train PSNR: {train_psnr_epoch:.2f} dB | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val PSNR: {val_psnr_epoch:.2f} dB"
        )

        if val_psnr_epoch > best_val_psnr:
            best_val_psnr = val_psnr_epoch
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_psnr": history["val_psnr"].copy(),
                    "config": {
                        "lr": LR,
                        "batch_size": BATCH_SIZE,
                        "epochs": epochs,
                        "sigma": SIGMA,
                    },
                },
                output_dir / f"{model_name}.pt",
            )

    return history


params_A = count_parameters(DeblurCNN())
params_B = count_parameters(DeblurCNN_BN())
params_C = count_parameters(DeblurDeep5())
print(
    f"模型已定義：DeblurCNN ({params_A:,} params), "
    f"DeblurCNN_BN ({params_B:,} params), "
    f"DeblurDeep5 ({params_C:,} params)"
)
print("統一訓練函數 train_model 已定義。")


<hr>

<strong><font color="darkgoldenrod">實驗開始</font></strong>

以下實驗依序進行資料集準備、三種卷積模型訓練、預訓練權重展示、Set5 與 Set14 測試評估，以及自訂影像定性比較。此順序使模型先在相同訓練條件下取得可比較的驗證曲線，再透過公開測試影像與單張自選影像檢查其泛化能力與視覺復原品質。


<hr>

<strong><font color="darkgoldenrod">資料集準備與視覺化</font></strong>

本節從 T91 與 General100 收集所有訓練影像，並將每張影像對應為 10 個 64×64 patch sampling units，再進行 9:1 的訓練與驗證切分。每次取樣時，資料集會先隨機裁切清晰 patch，再以 σ = 3 的高斯模糊即時生成輸入影像，因此磁碟上不需要額外保存模糊版本。視覺化部分將直接從訓練資料集取得前四筆樣本，並並排呈現模糊 patch 與對應的清晰 patch，以確認資料流程與模糊強度。


In [ ]:
image_extensions = {".png", ".jpg", ".jpeg", ".bmp"}
dir_counts = {}
image_paths = []
for train_dir in TRAIN_DIRS:
    paths = sorted(
        path
        for path in train_dir.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    dir_counts[train_dir.name] = len(paths)
    image_paths.extend(paths)

image_paths = sorted(image_paths)
patch_image_paths = [path for path in image_paths for _ in range(N_PATCHES)]
train_patch_paths, val_patch_paths = train_test_split(
    patch_image_paths,
    test_size=0.1,
    random_state=RANDOM_STATE,
    shuffle=True,
)

base_transform = transforms.ToTensor()
train_data = DeblurPatchDataset(train_patch_paths, PATCH_SIZE, 1, SIGMA, base_transform)
val_data = DeblurPatchDataset(val_patch_paths, PATCH_SIZE, 1, SIGMA, base_transform)

loader_workers = 0 if device == "mps" else 2
pin_memory = device == "cuda"
train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=loader_workers,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_data,
    batch_size=1,
    shuffle=False,
    num_workers=loader_workers,
    pin_memory=pin_memory,
)

print(
    f"總影像數量: {len(image_paths)} "
    f"(T91: {dir_counts.get('T91', 0)}, General100: {dir_counts.get('General100', 0)})"
)
print(f"訓練集 patches: {len(train_data)} | 驗證集 patches: {len(val_data)}")
print(f"TrainLoader 批次數: {len(train_loader)} | ValLoader 批次數: {len(val_loader)}")

fig, axes = plt.subplots(4, 2, figsize=(10, 18))
for row in range(4):
    blur_tensor, sharp_tensor = train_data[row]
    blur_img = blur_tensor.permute(1, 2, 0).numpy()
    sharp_img = sharp_tensor.permute(1, 2, 0).numpy()

    axes[row, 0].imshow(np.clip(blur_img, 0, 1))
    axes[row, 0].set_title("Blurred (σ=3)")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(np.clip(sharp_img, 0, 1))
    axes[row, 1].set_title("Sharp (Ground Truth)")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()


<hr>

<strong><font color="darkgoldenrod">Model A 訓練：DeblurCNN</font></strong>

Model A 採用 DeblurCNN，為 SRCNN 論文（Dong et al., 2014）精神下的三層卷積基線架構。第一層使用 9×9 kernel 進行較大感受野的特徵提取，第二層使用 1×1 kernel 進行非線性映射，第三層使用 5×5 kernel 進行影像重建。此模型與教授示範版本保持相同架構，因此可作為後續 BatchNorm 與更深網路設計的比較基準；其他模型的改動皆可視為在此三層 SRCNN 基礎上的穩定化或表達能力擴充。


In [ ]:
model_A = DeblurCNN().to(device)
results_A = train_model(
    model_A,
    "modelA_DeblurCNN",
    train_loader,
    val_loader,
    NUM_EPOCHS,
    device,
    OUTPUT_DIR,
    CosineAnnealingLR,
    {"T_max": NUM_EPOCHS},
)

epochs_A = range(1, len(results_A["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_A, results_A["train_loss"], label="Train Loss")
axes[0].plot(epochs_A, results_A["val_loss"], label="Val Loss")
axes[0].set_title("Model A: Loss vs Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_A, results_A["train_psnr"], label="Train PSNR")
axes[1].plot(epochs_A, results_A["val_psnr"], label="Val PSNR")
axes[1].set_title("Model A: PSNR vs Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "modelA_curves.png", dpi=150, bbox_inches="tight")
plt.show()

best_A = max(results_A["val_psnr"])
best_epoch_A = results_A["val_psnr"].index(best_A) + 1
print(f"Model A 最佳驗證 PSNR: {best_A:.2f} dB (Epoch {best_epoch_A}/{NUM_EPOCHS})")


Model A（DeblurCNN）於 50 個訓練週期後，訓練集 PSNR 達到 ___ dB，驗證集最佳 PSNR 為 ___ dB（第 ___ 個 epoch）。從損失曲線可觀察到，損失值在前 ___ 個 epoch 快速下降後趨於平緩，顯示模型已充分收斂。PSNR 曲線呈現 ___ 的趨勢，訓練集與驗證集之間的差距約為 ___ dB，___ 顯著的過擬合現象。


<hr>

<strong><font color="darkgoldenrod">Model B 訓練：DeblurCNN_BN</font></strong>

Model B 採用 DeblurCNN_BN，在三層卷積結構中加入 Batch Normalization（Ioffe & Szegedy, 2015），其作用在於標準化每層輸出的分布，使後續層接收的 activation 均值與變異數較為穩定，進而提升訓練穩定性並有助於梯度有效傳播。相較於 Model A，Model B 將 conv1 的 padding 調整為 4，conv2 的 padding 調整為 0，使空間尺寸在各層前後保持一致。<span style="color: crimson; font-weight: 600;">此模型與教授示範版本的關鍵差異，是使用 BatchNorm 進行標準化，而不是採用 residual connection。</span>


In [ ]:
model_B = DeblurCNN_BN().to(device)
results_B = train_model(
    model_B,
    "modelB_DeblurCNN_BN",
    train_loader,
    val_loader,
    NUM_EPOCHS,
    device,
    OUTPUT_DIR,
    CosineAnnealingLR,
    {"T_max": NUM_EPOCHS},
)

epochs_B = range(1, len(results_B["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_B, results_B["train_loss"], label="Train Loss")
axes[0].plot(epochs_B, results_B["val_loss"], label="Val Loss")
axes[0].set_title("Model B: Loss vs Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_B, results_B["train_psnr"], label="Train PSNR")
axes[1].plot(epochs_B, results_B["val_psnr"], label="Val PSNR")
axes[1].set_title("Model B: PSNR vs Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "modelB_curves.png", dpi=150, bbox_inches="tight")
plt.show()

best_B = max(results_B["val_psnr"])
best_epoch_B = results_B["val_psnr"].index(best_B) + 1
print(f"Model B 最佳驗證 PSNR: {best_B:.2f} dB (Epoch {best_epoch_B}/{NUM_EPOCHS})")


Model B（DeblurCNN_BN）加入 Batch Normalization 後，驗證集最佳 PSNR 為 ___ dB，與 Model A 相差 ___ dB。收斂速度 ___ Model A，損失曲線在第 ___ 個 epoch 附近出現明顯的轉折點。這一結果 ___ 了 BatchNorm 在影像復原任務中能夠加速訓練的假設。


<hr>

<strong><font color="darkgoldenrod">Model C 訓練：DeblurDeep5</font></strong>

Model C 採用 DeblurDeep5，將網路深度由三層增加至五層，以提升模型對複雜局部模糊結構的表達能力。此架構的 channel 配置為 3→64→128→64→32→3，形成先擴展後壓縮的 bottleneck-like 結構；所有卷積層皆使用 3×3 kernel 與適當 padding，使輸入與輸出維持相同空間尺寸。<span style="color: crimson; font-weight: 600;">相較於教授示範中的 DeblurSuperResCNN，本模型深度增加至五層，取消 feature concatenation，輸出層不使用 Sigmoid，並以均勻 3×3 kernel 取代混合 kernel 尺寸。</span>不在輸出層使用 Sigmoid 的目的，是避免 MSE 損失下過早壓縮輸出範圍而限制梯度流。


In [ ]:
model_C = DeblurDeep5().to(device)
results_C = train_model(
    model_C,
    "modelC_DeblurDeep5",
    train_loader,
    val_loader,
    NUM_EPOCHS,
    device,
    OUTPUT_DIR,
    CosineAnnealingLR,
    {"T_max": NUM_EPOCHS},
)

epochs_C = range(1, len(results_C["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_C, results_C["train_loss"], label="Train Loss")
axes[0].plot(epochs_C, results_C["val_loss"], label="Val Loss")
axes[0].set_title("Model C: Loss vs Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_C, results_C["train_psnr"], label="Train PSNR")
axes[1].plot(epochs_C, results_C["val_psnr"], label="Val PSNR")
axes[1].set_title("Model C: PSNR vs Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "modelC_curves.png", dpi=150, bbox_inches="tight")
plt.show()

best_C = max(results_C["val_psnr"])
best_epoch_C = results_C["val_psnr"].index(best_C) + 1
print(f"Model C 最佳驗證 PSNR: {best_C:.2f} dB (Epoch {best_epoch_C}/{NUM_EPOCHS})")


Model C（DeblurDeep5）以 5 層架構訓練後，驗證集最佳 PSNR 達 ___ dB，在三個模型中排名 ___。相較於 Model A 的基線，增加網路深度帶來了 ___ dB 的 PSNR 改善（或下降）。從訓練曲線可觀察到，較深的模型 ___ 更多的 epoch 才達到穩定收斂，這與較大的參數量（約 ___ 個可訓練參數）有關。


<hr>

<strong><font color="darkgoldenrod">預訓練模型展示</font></strong>

本節載入三個已儲存的最佳 checkpoint，並對每個模型額外訓練 2 個 epoch，以表格形式呈現 Train PSNR 與 Val PSNR 的延續情形。此處只執行短時間的示範訓練，目的並非重新訓練模型，而是確認已儲存權重能被正確載入，且模型可在相同資料流程上繼續優化，從而驗證訓練流程與權重保存機制的可復現性。


In [ ]:
demo_results = []
demo_output_dir = OUTPUT_DIR / "demo_runs"
demo_output_dir.mkdir(parents=True, exist_ok=True)

for model_name, model_cls in [
    ("modelA_DeblurCNN", DeblurCNN),
    ("modelB_DeblurCNN_BN", DeblurCNN_BN),
    ("modelC_DeblurDeep5", DeblurDeep5),
]:
    checkpoint_path = OUTPUT_DIR / f"{model_name}.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = model_cls().to(device)
    model.load_state_dict(checkpoint["model_state_dict"])

    demo_history = train_model(
        model,
        f"demo_{model_name}",
        train_loader,
        val_loader,
        DEMO_EPOCHS,
        device,
        demo_output_dir,
        CosineAnnealingLR,
        {"T_max": DEMO_EPOCHS},
    )

    demo_results.append(
        {
            "Model Name": model_name,
            "Pre-trained Epochs": checkpoint.get("epoch", np.nan),
            "Demo Epoch 1 Train PSNR": demo_history["train_psnr"][0],
            "Demo Epoch 1 Val PSNR": demo_history["val_psnr"][0],
            "Demo Epoch 2 Train PSNR": demo_history["train_psnr"][1],
            "Demo Epoch 2 Val PSNR": demo_history["val_psnr"][1],
        }
    )

demo_df = pd.DataFrame(demo_results)
display(demo_df)


三個預訓練模型於載入後各執行 2 個額外 epoch。Model A 的 Demo Epoch 2 驗證 PSNR 為 ___ dB，Model B 為 ___ dB，Model C 為 ___ dB。PSNR 數值與原本最佳 checkpoint 相比，___ 模型出現小幅提升，___ 模型則趨於穩定，顯示模型已大致收斂。此結果驗證了預訓練權重的可復現性：在相同資料上繼續訓練，模型能夠在既有的性能基礎上穩定延伸。


<hr>

<strong><font color="darkgoldenrod">Set5 與 Set14 測試評估</font></strong>

本節使用 Set5 與 Set14 共 19 張公開測試影像，評估三個模型在訓練資料之外的去模糊能力。Set5 與 Set14 長期被超解析度與影像復原文獻作為標準 benchmark，因此能提供比單一訓練驗證切分更具可比較性的測試情境。每個模型會產生一份 19 列 × 3 欄的影像對照圖，依序呈現原始清晰圖、動態模糊圖與模型復原圖，並在標題中標示模糊影像與復原影像相對於清晰影像的 PSNR。


In [ ]:
def image_to_tensor(img_rgb: np.ndarray) -> torch.Tensor:
    img_float = img_rgb.astype(np.float32) / 255.0
    return torch.from_numpy(img_float).permute(2, 0, 1).unsqueeze(0).float()


def tensor_to_image(tensor: torch.Tensor) -> np.ndarray:
    img = tensor.squeeze(0).detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1)


def load_trained_model(model_cls, model_name: str) -> nn.Module:
    checkpoint_path = OUTPUT_DIR / f"{model_name}.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = model_cls().to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model


def collect_test_paths() -> list[Path]:
    test_paths = []
    for test_dir in [SET5_DIR, SET14_DIR]:
        paths = sorted(
            path
            for path in test_dir.iterdir()
            if path.is_file() and path.suffix.lower() in image_extensions
        )
        test_paths.extend(paths)
    return sorted(test_paths, key=lambda path: (path.parent.name, path.name))


def evaluate_model_on_sets(model_cls, model_name: str, display_name: str, output_name: str) -> pd.DataFrame:
    model = load_trained_model(model_cls, model_name)
    test_paths = collect_test_paths()
    if len(test_paths) != 19:
        raise ValueError(f"Expected 19 test images, found {len(test_paths)}")

    fig, axes = plt.subplots(19, 3, figsize=(12, 4 * 19))
    records = []

    for row, img_path in enumerate(test_paths):
        sharp_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
        if sharp_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        sharp_np = cv2.cvtColor(sharp_bgr, cv2.COLOR_BGR2RGB)
        blur_np = apply_blur(sharp_np, SIGMA)

        sharp_tensor = image_to_tensor(sharp_np)
        blur_tensor = image_to_tensor(blur_np)
        with torch.no_grad():
            deblur_tensor = model(blur_tensor.to(device)).cpu().clamp(0, 1)

        psnr_blur = psnr(blur_tensor, sharp_tensor)
        psnr_deblur = psnr(deblur_tensor, sharp_tensor)
        deblur_np = tensor_to_image(deblur_tensor)

        axes[row, 0].imshow(np.clip(sharp_tensor.squeeze(0).permute(1, 2, 0).numpy(), 0, 1))
        axes[row, 0].set_title(f"{img_path.name}\nSharp")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(np.clip(blur_tensor.squeeze(0).permute(1, 2, 0).numpy(), 0, 1))
        axes[row, 1].set_title(f"Blurred | PSNR: {psnr_blur:.2f} dB")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(deblur_np)
        axes[row, 2].set_title(f"Deblurred | PSNR: {psnr_deblur:.2f} dB")
        axes[row, 2].axis("off")

        records.append(
            {
                "dataset": img_path.parent.name,
                "image_name": img_path.name,
                "blur_psnr": psnr_blur,
                "deblur_psnr": psnr_deblur,
            }
        )

    plt.suptitle(f"{display_name} — Set5 & Set14 Evaluation", fontsize=16, y=1.001)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / output_name, dpi=100, bbox_inches="tight")
    plt.show()

    result_df = pd.DataFrame(records)
    set5_mean = result_df.loc[result_df["dataset"] == "Set5", "deblur_psnr"].mean()
    set14_mean = result_df.loc[result_df["dataset"] == "Set14", "deblur_psnr"].mean()
    overall_mean = result_df["deblur_psnr"].mean()
    print(
        f"{display_name} Set5 平均 PSNR: {set5_mean:.2f} dB | "
        f"Set14 平均 PSNR: {set14_mean:.2f} dB | "
        f"總體平均: {overall_mean:.2f} dB"
    )
    return result_df


set_results_A = evaluate_model_on_sets(
    DeblurCNN,
    "modelA_DeblurCNN",
    "Model A (DeblurCNN)",
    "modelA_set5set14_grid.png",
)


In [ ]:
set_results_B = evaluate_model_on_sets(
    DeblurCNN_BN,
    "modelB_DeblurCNN_BN",
    "Model B (DeblurCNN_BN)",
    "modelB_set5set14_grid.png",
)


In [ ]:
set_results_C = evaluate_model_on_sets(
    DeblurDeep5,
    "modelC_DeblurDeep5",
    "Model C (DeblurDeep5)",
    "modelC_set5set14_grid.png",
)


在 Set5 測試集上，三個模型的平均 PSNR 分別為 Model A：___ dB、Model B：___ dB、Model C：___ dB。Set14 上的結果則為 Model A：___ dB、Model B：___ dB、Model C：___ dB。整體而言，___ 模型在兩個測試集上均表現最佳。值得注意的是，部分影像（如 ___ ）的去模糊效果較佳，這可能與其 ___ 的紋理特性有關。相較於模糊圖，三個模型平均提升了 ___ dB 的 PSNR，顯示卷積神經網路在去除高斯模糊方面具備一定的學習能力。


<hr>

<strong><font color="darkgoldenrod">自訂影像測試</font></strong>

本節使用自選影像進行定性測試，影像位置由全域常數中的 `CUSTOM_IMG_PATH` 決定，因此可在不改動後續流程的情況下替換測試圖像。測試流程先讀入清晰影像，再以相同的高斯模糊函數動態產生模糊版本，接著分別載入三個模型的最佳 checkpoint 進行去模糊推論，最後將清晰圖、模糊圖與三個模型輸出並排呈現，使定量 PSNR 與視覺品質能同步比較。


In [ ]:
sharp_bgr = cv2.imread(str(CUSTOM_IMG_PATH), cv2.IMREAD_COLOR)
if sharp_bgr is None:
    raise FileNotFoundError(f"Cannot read image: {CUSTOM_IMG_PATH}")
sharp_np = cv2.cvtColor(sharp_bgr, cv2.COLOR_BGR2RGB)
blur_np = apply_blur(sharp_np, SIGMA)

sharp_tensor = image_to_tensor(sharp_np)
blur_tensor = image_to_tensor(blur_np)

custom_models = [
    ("Model A", load_trained_model(DeblurCNN, "modelA_DeblurCNN")),
    ("Model B", load_trained_model(DeblurCNN_BN, "modelB_DeblurCNN_BN")),
    ("Model C", load_trained_model(DeblurDeep5, "modelC_DeblurDeep5")),
]

custom_outputs = []
with torch.no_grad():
    for label, model in custom_models:
        deblur_tensor = model(blur_tensor.to(device)).cpu().clamp(0, 1)
        custom_outputs.append((label, deblur_tensor, psnr(deblur_tensor, sharp_tensor)))

blur_score = psnr(blur_tensor, sharp_tensor)
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(np.clip(sharp_tensor.squeeze(0).permute(1, 2, 0).numpy(), 0, 1))
axes[0].set_title("Sharp (Ground Truth)")
axes[0].axis("off")

axes[1].imshow(np.clip(blur_tensor.squeeze(0).permute(1, 2, 0).numpy(), 0, 1))
axes[1].set_title(f"Blurred\nPSNR: {blur_score:.2f} dB")
axes[1].axis("off")

for axis, (label, deblur_tensor, score) in zip(axes[2:], custom_outputs):
    axis.imshow(tensor_to_image(deblur_tensor))
    axis.set_title(f"{label} Deblurred\nPSNR: {score:.2f} dB")
    axis.axis("off")

plt.suptitle(f"Custom Image Test: {CUSTOM_IMG_PATH.name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "custom_test.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"影像路徑: {CUSTOM_IMG_PATH}")
print(f"Blurred   PSNR: {blur_score:.2f} dB")
for label, _, score in custom_outputs:
    print(f"{label:<9} PSNR: {score:.2f} dB")


針對自選影像（___ ），模糊化後的 PSNR 為 ___ dB。Model A 去模糊後恢復至 ___ dB，Model B 恢復至 ___ dB，Model C 恢復至 ___ dB。從視覺結果觀察，___ 模型產生的去模糊影像在 ___ 方面（如邊緣銳利度、色彩保真度）表現較為突出，而 ___ 區域仍可見殘留的模糊或人工假影。此結果與 Set5/Set14 的定量評估 ___ 一致。


In [ ]:
summary_rows = []
for model_name, model_cls, result_key in [
    ("DeblurCNN", DeblurCNN, results_A),
    ("DeblurCNN_BN", DeblurCNN_BN, results_B),
    ("DeblurDeep5", DeblurDeep5, results_C),
]:
    best_val_psnr = max(result_key["val_psnr"])
    summary_rows.append(
        {
            "model_name": model_name,
            "n_params": count_parameters(model_cls()),
            "best_val_psnr": best_val_psnr,
            "final_val_psnr": result_key["val_psnr"][-1],
            "best_epoch": result_key["val_psnr"].index(best_val_psnr) + 1,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "results_summary.csv"
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print(f"結果已存至: {summary_path}")


本實驗以 SRCNN 為基礎，設計並比較三種影像去模糊卷積神經網路架構。在 T91 與 General100 合計 191 張訓練影像、64×64 隨機 patch 策略下，三個模型於驗證集上的最終 PSNR 分別為 ___ dB（DeblurCNN）、___ dB（DeblurCNN_BN）與 ___ dB（DeblurDeep5）。整體而言，___ 模型取得最佳結果，相較於基線提升了 ___ dB。

從模型設計觀點來看，BatchNorm 的加入使 Model B 相較於 Model A 在收斂速度與訓練穩定性上呈現 ___ 的變化，而更深的 Model C 則以較大的參數量換取 ___ 的表達能力。與教授示範程式相比，本實驗同時改變資料來源、patch crop 策略、模糊產生方式與學習率排程，因此結果反映的不只是單一模型架構差異，也包含資料流程與訓練配置的系統性調整。實驗仍有若干限制，包括訓練資料規模有限、模糊型態僅限於高斯模糊，以及損失函數只採用像素層級 MSE，尚未納入 perceptual loss 或 adversarial loss。未來若要提升真實影像去模糊能力，可進一步擴充資料來源、加入不同模糊 kernel 與雜訊條件，並比較更深的 residual 或 attention-based 架構。
